# Custom Architectures (Model A & Model B)

Two CNNs for the RealWaste dataset (9 classes, 64x64x3 RGB input, pixels normalized to [0, 1]):

- **Model A** — Standard CNN: plain `Conv2D` + ReLU + `MaxPool2D` blocks, then `Dense` + Softmax.
- **Model B** — Lightweight CNN: standard convs replaced with **Depthwise Separable Convolutions**, strictly under **100,000** trainable parameters.

Implemented in PyTorch (`models.py`) since TensorFlow does not support the Python version in this environment; PyTorch is an accepted alternative per the assignment brief.

In [1]:
import torch
from torchinfo import summary

from models import ModelA, ModelB, count_params, IMG_SIZE, NUM_CLASSES

torch.manual_seed(42)

INPUT_SHAPE = (1, 3, IMG_SIZE, IMG_SIZE)  # (batch, channels, height, width)
print(f"Input shape: {INPUT_SHAPE}, num_classes: {NUM_CLASSES}")

Input shape: (1, 3, 64, 64), num_classes: 9


## Model A: Standard CNN

In [2]:
model_a = ModelA(num_classes=NUM_CLASSES)
summary(model_a, input_size=INPUT_SHAPE)

Layer (type:depth-idx)                   Output Shape              Param #
ModelA                                   [1, 9]                    --
├─Conv2d: 1-1                            [1, 16, 64, 64]           448
├─ReLU: 1-2                              [1, 16, 64, 64]           --
├─MaxPool2d: 1-3                         [1, 16, 32, 32]           --
├─Conv2d: 1-4                            [1, 32, 32, 32]           4,640
├─ReLU: 1-5                              [1, 32, 32, 32]           --
├─MaxPool2d: 1-6                         [1, 32, 16, 16]           --
├─Conv2d: 1-7                            [1, 64, 16, 16]           18,496
├─ReLU: 1-8                              [1, 64, 16, 16]           --
├─MaxPool2d: 1-9                         [1, 64, 8, 8]             --
├─Flatten: 1-10                          [1, 4096]                 --
├─Linear: 1-11                           [1, 9]                    36,873
├─Softmax: 1-12                          [1, 9]                    --
Tot

### Model A — parameter calculation (by hand)

For a `Conv2D` layer: `params = (kernel_h * kernel_w * in_channels + 1) * out_channels` (the `+1` is the bias per output filter).
For a `Dense`/`Linear` layer: `params = in_features * out_features + out_features`.

| Layer | Operation | Output shape | Params |
|---|---|---|---|
| Conv1 | 3x3, 3->16, ReLU | 64x64x16 | (3*3*3 + 1) * 16 = **448** |
| MaxPool | 2x2 | 32x32x16 | 0 |
| Conv2 | 3x3, 16->32, ReLU | 32x32x32 | (3*3*16 + 1) * 32 = **4,640** |
| MaxPool | 2x2 | 16x16x32 | 0 |
| Conv3 | 3x3, 32->64, ReLU | 16x16x64 | (3*3*32 + 1) * 64 = **18,496** |
| MaxPool | 2x2 | 8x8x64 | 0 |
| Flatten | - | 4096 | 0 |
| Dense | 4096 -> 9, Softmax | 9 | 4096*9 + 9 = **36,873** |
| **Total** | | | **60,457** |

This matches the `torchinfo.summary()` output above. Note the Dense layer alone (36,873) accounts for over 60% of Model A's parameters — flattening a spatial feature map straight into a big fully-connected layer is expensive, which is exactly the kind of cost an edge-constrained design needs to avoid.

## Model B: Lightweight CNN (Depthwise Separable Convolutions)

In [3]:
model_b = ModelB(num_classes=NUM_CLASSES)
summary(model_b, input_size=INPUT_SHAPE)

Layer (type:depth-idx)                   Output Shape              Param #
ModelB                                   [1, 9]                    --
├─Conv2d: 1-1                            [1, 16, 64, 64]           448
├─ReLU: 1-2                              [1, 16, 64, 64]           --
├─MaxPool2d: 1-3                         [1, 16, 32, 32]           --
├─SeparableConv2d: 1-4                   [1, 32, 32, 32]           --
│    └─Conv2d: 2-1                       [1, 16, 32, 32]           160
│    └─Conv2d: 2-2                       [1, 32, 32, 32]           544
├─ReLU: 1-5                              [1, 32, 32, 32]           --
├─MaxPool2d: 1-6                         [1, 32, 16, 16]           --
├─SeparableConv2d: 1-7                   [1, 64, 16, 16]           --
│    └─Conv2d: 2-3                       [1, 32, 16, 16]           320
│    └─Conv2d: 2-4                       [1, 64, 16, 16]           2,112
├─ReLU: 1-8                              [1, 64, 16, 16]           --
├─MaxPoo

In [4]:
params_a = count_params(model_a)
params_b = count_params(model_b)
print(f"Model A total trainable params: {params_a:,}")
print(f"Model B total trainable params: {params_b:,}")
assert params_b < 100_000, "Model B must stay under the 100k parameter budget"
print("Constraint satisfied: Model B < 100,000 parameters.")

Model A total trainable params: 60,457
Model B total trainable params: 40,457
Constraint satisfied: Model B < 100,000 parameters.


### Model B — parameter calculation (by hand)

A **depthwise separable conv** splits a standard 3x3 conv into two cheaper steps:
1. **Depthwise**: one 3x3 filter per input channel (no channel mixing). `params = (3*3*1 + 1) * in_channels` (`groups=in_channels`).
2. **Pointwise**: a 1x1 conv that mixes channels. `params = (1*1*in_channels + 1) * out_channels`.

| Layer | Operation | Output shape | Params |
|---|---|---|---|
| Conv1 (standard) | 3x3, 3->16, ReLU | 64x64x16 | (3*3*3 + 1) * 16 = **448** |
| MaxPool | 2x2 | 32x32x16 | 0 |
| SepConv2 (depthwise) | 3x3, 16 groups | 32x32x16 | (3*3*1 + 1) * 16 = **160** |
| SepConv2 (pointwise) | 1x1, 16->32 | 32x32x32 | (1*1*16 + 1) * 32 = **544** |
| MaxPool | 2x2 | 16x16x32 | 0 |
| SepConv3 (depthwise) | 3x3, 32 groups | 16x16x32 | (3*3*1 + 1) * 32 = **320** |
| SepConv3 (pointwise) | 1x1, 32->64 | 16x16x64 | (1*1*32 + 1) * 64 = **2,112** |
| MaxPool | 2x2 | 8x8x64 | 0 |
| Flatten | - | 4096 | 0 |
| Dense | 4096 -> 9, Softmax | 9 | 4096*9 + 9 = **36,873** |
| **Total** | | | **40,457** |

Compare the two conv blocks directly: a standard 32->64 conv (Conv3 in Model A) costs 18,496 params, while the depthwise-separable equivalent (SepConv3) costs only 320 + 2,112 = 2,432 — roughly an **8x reduction** for that block. This is the standard depthwise-separable saving (`~ 1/out_channels + 1/kernel_area` of the original cost).

The first conv layer (3 input channels) is kept as a standard conv on purpose: with only 3 input channels there is almost nothing to "separate" — a depthwise pass over 3 channels barely saves parameters but still adds an extra pointwise layer, so it's not worth the complexity there.

Note the Dense layer (36,873) is now the *dominant* cost in Model B, not the convolutions — this is the classic edge-CNN trade-off: once convs are made cheap with depthwise separable layers, the flatten+dense head becomes the new bottleneck, which is why real MobileNet-style architectures use Global Average Pooling instead of a big Flatten before the classifier (explored for the SOTA models in the transfer learning notebook).

## Hardware-aware justification: activation functions

**ReLU** (`max(0, x)`) is used after every conv layer in both models, instead of `sigmoid` or `tanh`, for edge-hardware reasons:

- **Compute cost**: `sigmoid`/`tanh` require an exponential (`e^x`), which is a multi-cycle operation on a CPU/microcontroller with no dedicated hardware exponential unit. ReLU is a single comparison (`x > 0 ? x : 0`) — effectively free, and trivially vectorizable/quantizable.
- **No expensive division**: `sigmoid = 1 / (1 + e^-x)` also needs a division; ReLU needs neither.
- **Sparsity**: ReLU zeroes out negative activations, producing sparse feature maps. Sparse data compresses and skips-computation better on constrained hardware (e.g. many edge accelerators can skip zero-valued MACs).
- **Gradient behavior**: sigmoid/tanh saturate and vanish gradients for large |x|, slowing training; ReLU's gradient is either 0 or 1, cheap to compute and avoids vanishing gradients for positive activations.

**Softmax** is still used, but only once, at the very output layer (9-way class probabilities), not inside the network — it's needed there to turn raw scores into a valid probability distribution for classification, and paying one exponential per output class (9 exponentials) is negligible compared to paying it after every conv layer across a whole feature map (thousands of activations).